<a href="https://colab.research.google.com/github/amoulabensalem/ALSTM-CA/blob/main/Copie_de_NCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

drive.mount('/content/gdrive')

Mounted at /content/gdrive


# The Models with Filters 3*3


```
# Ce texte est au format code
```



In [ ]:
"""Model definitions."""


import torch
import torch.nn as nn
import torch.nn.functional as F

class UrbanNCA(nn.Module):
    """
    Neural Cellular Automata model for urban growth simulation.
    """
    def __init__(self, channels=16):
        super(UrbanNCA, self).__init__()
        self.channels = channels

        # 1. Perception Layer
        # Fixed 3x3 depthwise convolutions for Identity, Sobel X, and Sobel Y

        # Define the 3x3 filters
        ident = torch.tensor([[0., 0., 0.],
                              [0., 1., 0.],
                              [0., 0., 0.]])
        # Sobel filters for edge detection in X and Y
        sobel_x = torch.tensor([[-1., 0., 1.],
                                [-2., 0., 2.],
                                [-1., 0., 1.]]) / 8.0
        sobel_y = torch.tensor([[-1., -2., -1.],
                                [ 0.,  0.,  0.],
                                [ 1.,  2.,  1.]]) / 8.0
        # 🌟 NEW: Define the 8-neighbor counting filter
        neighbor_count = torch.tensor([[1., 1., 1.],
                                       [1., 0., 1.],
                                       [1., 1., 1.]] ) /8.0 # Normalize to keep values between 0 and 1
        # Stack filters: shape (4, 1, 3, 3)
        filters = torch.stack([ident, sobel_x, sobel_y, neighbor_count]).unsqueeze(1)

        # Repeat for all channels to create depthwise convolution weights
        # Shape: (channels * 3, 1, 3, 3)
        filters = filters.repeat(channels, 1, 1, 1)

        self.perception = nn.Conv2d(
            in_channels=channels,
            out_channels=channels * 4,
            kernel_size=3,
            padding=1,
            groups=channels,
            bias=False
        )
        # Set weights and freeze them to make them non-trainable
        self.perception.weight.data = filters
        self.perception.weight.requires_grad = False

        # 2. Update Network
        # Small MLP with 1x1 convolutions
        self.update_net = nn.Sequential(
            nn.Conv2d(channels * 4, 128, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(128, channels, kernel_size=1)
        )

        # Initialize final Conv2d weights and biases to zero
        # This ensures the initial update step does nothing
        nn.init.zeros_(self.update_net[-1].weight)
        nn.init.zeros_(self.update_net[-1].bias)

    def forward(self, x, steps=1):
        """
        Forward pass of the NCA.
        x: Initial state tensor of shape (B, 16, H, W)
        steps: Number of simulation steps to run dynamically
        """
        # 1. Create a logical mask for the channels (1.0 = dynamic, 0.0 = static)
        # Shape: (1, 16, 1, 1) to broadcast across the whole batch/grid
        channel_mask = torch.ones(1, self.channels, 1, 1, device=x.device)
        static_channels = [5, 6, 7, 8,]
        channel_mask[:, static_channels, :, :] = 0.0

        # 2. Extract the water mask (0.0 = buildable, 1.0 = water)
        water_mask = x[:, 5:6, :, :].clone()

        for _ in range(steps):
            # Evaluate perception layer (neighbors)
            perception_out = self.perception(x)

            # Compute suggested change (dx) via update network
            dx = self.update_net(perception_out)

            # --- APPLY GEOGRAPHIC RULES TO dx (Gradient-Safe!) ---

            # A. Overwriting Rule: Force dx to 0 for all static channels
            dx = dx * channel_mask

            # B. Water Masking: Force dx to 0 for the Urban channel (0) if there is water
            # (water_mask == 0.0) returns 1.0 for dry land, 0.0 for water
            dx[:, 0:1, :, :] = dx[:, 0:1, :, :] * (water_mask == 0.0).float()

            # -----------------------------------------------------

            # Stochastic update: approximately 50% of cells per step
            update_mask = (torch.rand(x.shape[0], 1, x.shape[2], x.shape[3], device=x.device) < 0.5).float()

            # Apply dx to original state using the stochastic mask
            x = x + dx * update_mask

        return x



In [ ]:
print('Listing contents of /content/gdrive:')
!ls -R /content/gdrive

In [ ]:
import os
import numpy as np
import rasterio
import torch

# =====================================================================
# 1. HELPER FUNCTIONS
# =====================================================================
def load_tif(filepath):
    """Opens a single-band .tif file and reads it as a float32 numpy array."""
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Missing file: {filepath}")
    with rasterio.open(filepath) as src:
        return src.read(1).astype(np.float32)

def normalize(arr):
    min_val, max_val = np.min(arr), np.max(arr)
    if max_val == min_val: return np.zeros_like(arr)
    return (arr - min_val) / (max_val - min_val)

def invert(arr):
    return 1.0 - arr

# =====================================================================
# 2. LOAD 2000 DATA (THE STARTING POINT & DRIVERS)
# =====================================================================
# The Seed: use the real 2000 urban map stored in the inputs folder.
seed_map = load_tif("/content/gdrive/MyDrive/DatasetNCA/inputs/hn_ua_late1990s_mask.tif")

# The Land Cover Drivers (lc1 through lc7)
agri_drv         = load_tif("/content/gdrive/MyDrive/DatasetNCA/drivers/hn_1990_lc2.tif")
forest_drv       = load_tif("/content/gdrive/MyDrive/DatasetNCA/drivers/hn_1990_lc3.tif")
grass_drv        = load_tif("/content/gdrive/MyDrive/DatasetNCA/drivers/hn_1990_lc4.tif")
barren_drv       = load_tif("/content/gdrive/MyDrive/DatasetNCA/drivers/hn_1990_lc6.tif")
water_drv        = load_tif("/content/gdrive/MyDrive/DatasetNCA/drivers/hn_1990_lc7.tif")

# Infrastructure Drivers
subway_drv  = invert(normalize(load_tif("/content/gdrive/MyDrive/DatasetNCA/drivers/hn_sws_1990s.tif")))
highway_drv = invert(normalize(load_tif("/content/gdrive/MyDrive/DatasetNCA/drivers/hn_ic_1990s.tif")))
river_drv = invert(normalize(load_tif("/content/gdrive/MyDrive/DatasetNCA/drivers/hn_water_acc.tif")))

# =====================================================================
# 3. BUILD THE 16-CHANNEL SEED TENSOR (X_seed)
# =====================================================================
H, W = seed_map.shape
hidden_channels = [np.zeros((H, W), dtype=np.float32) for _ in range(7)]

all_layers = [
    seed_map.copy(),   # Channel 0: Dynamic Target (Starts as late 2000s layout)
    agri_drv,          # Channel 1: Dynamic
    forest_drv,        # Channel 2: Dynamic
    grass_drv,         # Channel 3: Dynamic
    barren_drv,        # Channel 4: Dynamic
    water_drv,         # Channel 5: Static Constraint (lc7)
    subway_drv,        # Channel 6: Static Driver (sws)
    highway_drv,       # Channel 7: Static Driver (ic)
    river_drv          # Channel 8: Static Constraint
] + hidden_channels     # Channels 9-15: Hidden Communication

# Stack and convert to PyTorch Tensor -> Shape: (1, 16, H, W)
X_seed = torch.tensor(np.stack(all_layers, axis=0), dtype=torch.float32).unsqueeze(0)

# =====================================================================
# 4. LOAD 2010 DATA (THE TARGET)
# =====================================================================
# Load the late 2010s target map from your target folder
target_map = load_tif("/content/gdrive/MyDrive/DatasetNCA/inputs/hn_ua_late2000s_mask.tif")

# Convert to PyTorch Tensor -> Shape: (1, 1, H, W)
Y_target = torch.tensor(target_map, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

print(f"Model Input (X_seed) Shape: {X_seed.shape}")
print(f"Loss Target (Y_target) Shape: {Y_target.shape}")

# Check if the starting map is identical to the target map
are_identical = torch.equal(X_seed[:, 0:1, :, :], Y_target)
print(f"Are the Seed and Target identical? {are_identical}")

# Look at the raw number of urban pixels in each
print(f"Urban pixels in Seed: {X_seed[:, 0:1, :, :].sum().item()}")
print(f"Urban pixels in Target: {Y_target.sum().item()}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class UrbanNCA(nn.Module):
    """
    Neural Cellular Automata model for urban growth simulation.
    """
    def __init__(self, channels=16):
        super(UrbanNCA, self).__init__()
        self.channels = channels

        # 1. Perception Layer
        # Fixed 5x5 depthwise convolutions for Identity, Sobel X, and Sobel Y

        # Define the 5x5 filters
        # Identity filter
        ident = torch.zeros(5, 5)
        ident[2, 2] = 1.

        # Sobel filters for edge detection in X and Y (true 5x5 implementation)
        # A common 5x5 Sobel approximation
        sobel_x = torch.tensor(
            [[-1., -2., 0., 2., 1.],
             [-2., -3., 0., 3., 2.],
             [-3., -5., 0., 5., 3.],
             [-2., -3., 0., 3., 2.],
             [-1., -2., 0., 2., 1.]]
        ) / 32.0 # Normalize

        sobel_y = torch.tensor(
            [[-1., -2., -3., -2., -1.],
             [-2., -3., -5., -3., -2.],
             [ 0.,  0.,  0.,  0.,  0.],
             [ 2.,  3.,  5.,  3.,  2.],
             [ 1.,  2.,  3.,  2.,  1.]]
        ) / 32.0 # Normalize

        # 🌟 NEW: Define the 8-neighbor counting filter (extended to 5x5)
        neighbor_count = torch.ones(5, 5)
        neighbor_count[2, 2] = 0. # Center pixel is 0
        neighbor_count = neighbor_count / 24.0 # Normalize by sum of active pixels (24 for 5x5)

        # Stack filters: shape (4, 1, 5, 5)
        filters = torch.stack([ident, sobel_x, sobel_y, neighbor_count]).unsqueeze(1)

        # Repeat for all channels to create depthwise convolution weights
        # Shape: (channels * 3, 1, 5, 5)
        filters = filters.repeat(channels, 1, 1, 1)

        self.perception = nn.Conv2d(
            in_channels=channels,
            out_channels=channels * 4,
            kernel_size=5, # Changed to 5
            padding=2,     # Changed to 2 for 5x5 kernel
            groups=channels,
            bias=False
        )
        # Set weights and freeze them to make them non-trainable
        self.perception.weight.data = filters
        self.perception.weight.requires_grad = False

        # 2. Update Network
        # Small MLP with 1x1 convolutions
        self.update_net = nn.Sequential(
            nn.Conv2d(channels * 4, 128, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(128, channels, kernel_size=1)
        )

        # Initialize final Conv2d weights and biases to zero
        # This ensures the initial update step does nothing
        nn.init.zeros_(self.update_net[-1].weight)
        nn.init.zeros_(self.update_net[-1].bias)

    def forward(self, x, steps=1):
        """
        Forward pass of the NCA.
        x: Initial state tensor of shape (B, 16, H, W)
        steps: Number of simulation steps to run dynamically
        """
        # 1. Create a logical mask for the channels (1.0 = dynamic, 0.0 = static)
        # Shape: (1, 16, 1, 1) to broadcast across the whole batch/grid
        channel_mask = torch.ones(1, self.channels, 1, 1, device=x.device)
        static_channels = [5, 6, 7, 8,]
        channel_mask[:, static_channels, :, :] = 0.0

        # 2. Extract the water mask (0.0 = buildable, 1.0 = water)
        water_mask = x[:, 5:6, :, :].clone()

        for _ in range(steps):
            # Evaluate perception layer (neighbors)
            perception_out = self.perception(x)

            # Compute suggested change (dx) via update network
            dx = self.update_net(perception_out)

            # --- APPLY GEOGRAPHIC RULES TO dx (Gradient-Safe!) ---

            # A. Overwriting Rule: Force dx to 0 for all static channels
            dx = dx * channel_mask

            # B. Water Masking: Force dx to 0 for the Urban channel (0) if there is water
            # (water_mask == 0.0) returns 1.0 for dry land, 0.0 for water
            dx[:, 0:1, :, :] = dx[:, 0:1, :, :] * (water_mask == 0.0).float()

            # -----------------------------------------------------

            # Stochastic update: approximately 80% of cells per step
            update_mask = (torch.rand(x.shape[0], 1, x.shape[2], x.shape[3], device=x.device) < 0.8).float()

            # Apply dx to original state using the stochastic mask
            x = x + dx * update_mask

        return x

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
# from src.datasets import X_seed  # Removed as X_seed and Y_target are already defined
# from src.models import UrbanNCA # Removed as UrbanNCA is already defined

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# 2. Limit CPU threads to prevent deadlocks and optimize performance
torch.set_num_threads(4)

# Create the outputs folder if it doesn't exist
os.makedirs("outputs", exist_ok=True)

# 1. Setup Device (Use GPU if you have one, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

# 2. Initialize Model, Optimizer, and Loss Function
model = UrbanNCA(channels=16).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

# 3. Move your Data to the Device
X_seed = X_seed.to(device)
Y_target = Y_target.to(device)

# 4. Training Parameters
epochs = 1000

print("Starting training...")
print("-" * 50)

best_loss = float('inf') # Initialize best_loss to infinity

for epoch in range(epochs):
    # Clear the old gradients from the last step
    optimizer.zero_grad()

    # Randomize simulation steps to ensure stable growth (e.g., between 40 and 60 steps)
    sim_steps = np.random.randint(70, 90)
    if epoch == 0:
        print(f"Running forward pass with {sim_steps} steps on CPU... (This might take a moment)")
    # Run the simulation forward
    # Remember: X_seed contains our 2000s base maps

    # 🧠 THE CRASH SOLUTION: Run the bulk of steps without tracking history
    with torch.no_grad():
        burn_in_steps = sim_steps
        x_state = X_seed.clone()
        x_state = model(x_state, steps=burn_in_steps)

    # Run only the final 4 steps with gradient tracking to keep CPU RAM completely safe
    final_state = model(x_state, steps=sim_steps)
    #final_state = model(X_seed, steps=sim_steps)

    # Extract ONLY Channel 0 (The predicted 2010 Urban Map)
    pred_2010_urban = final_state[:, 0:1, :, :]

     # Compare Channel 0 against the actual 2010 Target
    loss = criterion(pred_2010_urban, Y_target)

    # Backpropagation (Calculate how to fix the mistakes)
    loss.backward()

    # Gradient clipping (Crucial for NCA to prevent exploding gradients)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    # Update the network weights
    optimizer.step()

    # Save best model
    if loss.item() < best_loss:
        best_loss = loss.item()
        torch.save(model.state_dict(), "outputs/trained_nca_model.pth")
        print(f"Epoch [{epoch + 1}/{epochs}] | Steps: {sim_steps} | Loss: {loss.item():.6f} (Best loss. Model saved!)")
    else:
        # Print progress every 10 epochs
        print(f"Epoch [{epoch + 1}/{epochs}] | Steps: {sim_steps} | Loss: {loss.item():.6f}")

print("-" * 50)
print("Training Complete!")

# The best model has already been saved during training.
print("The best model (with the lowest loss) has been saved to outputs/trained_nca_model.pth")

In [ ]:
import os
import torch
import rasterio
import numpy as np

# Leverage your excellent project structure!
# from src.datasets import X_seed # Removed as X_seed is already defined
# from src.models import UrbanNCA # Removed as UrbanNCA is already defined

# =====================================================================
# 1. SETUP DEVICE
# =====================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Loading model on: {device}")

# =====================================================================
# 2. INITIALIZE MODEL & LOAD TRAINED WEIGHTS
# =====================================================================
# Rebuild the model architecture from your src folder
model = UrbanNCA(channels=16).to(device)

# Load the saved .pth weights file from your outputs directory
model.load_state_dict(torch.load("outputs/trained_nca_model.pth", map_location=device))

# Set model to evaluation mode (turns off training mechanics)
model.eval()

# =====================================================================
# 1. HELPER FUNCTIONS
# =====================================================================
def load_tif(filepath):
    """Opens a single-band .tif file and reads it as a float32 numpy array."""
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Missing file: {filepath}")
    with rasterio.open(filepath) as src:
        return src.read(1).astype(np.float32)

def normalize(arr):
    min_val, max_val = np.min(arr), np.max(arr)
    if max_val == min_val: return np.zeros_like(arr)
    return (arr - min_val) / (max_val - min_val)

def invert(arr):
    return 1.0 - arr

def prep_distance_fixed(filepath_1990, filepath_2000):
    """Normalizes the 1990 map using the exact scale of the 2000 map to prevent distribution shift."""
    # 1. Get the exact scale from the 2000 training map
    arr_2000 = load_tif(filepath_2000)
    fixed_min = np.nanmin(arr_2000)
    fixed_max = np.nanmax(arr_2000)

    # 2. Load the 1990 map
    arr_1990 = load_tif(filepath_1990)

    # 3. Scale 1990 using the 2000 min/max
    if fixed_max == fixed_min:
        return np.zeros_like(arr_1990)

    normalized = (arr_1990 - fixed_min) / (fixed_max - fixed_min)

    # 4. Clip values just in case 1990 had longer distances (prevent negative or >1 values)
    normalized = np.clip(normalized, 0.0, 1.0)

    # 5. Invert it so roads are hotspots
    inverted = 1.0 - normalized
    inverted[np.isnan(inverted)] = 0.0
    return inverted

# =====================================================================
# 2. LOAD 2000 DATA (THE STARTING POINT & DRIVERS)
# =====================================================================
# The Seed: use the real 2000 urban map stored in the inputs folder.
seed_2000 = load_tif("/content/gdrive/MyDrive/DatasetNCA/inputs/hn_ua_late2000s_mask.tif")

# The Land Cover Drivers (lc1 through lc7)
agri_drv         = load_tif("/content/gdrive/MyDrive/DatasetNCA/drivers/hn_2000_lc2.tif")
forest_drv       = load_tif("/content/gdrive/MyDrive/DatasetNCA/drivers/hn_2000_lc3.tif")
grass_drv        = load_tif("/content/gdrive/MyDrive/DatasetNCA/drivers/hn_2000_lc4.tif")
barren_drv       = load_tif("/content/gdrive/MyDrive/DatasetNCA/drivers/hn_2000_lc6.tif")
water_drv        = load_tif("/content/gdrive/MyDrive/DatasetNCA/drivers/hn_2000_lc7.tif")

# Infrastructure Drivers
subway_drv  = invert(normalize(load_tif("/content/gdrive/MyDrive/DatasetNCA/drivers/hn_sws_2000s.tif")))
highway_drv = invert(normalize(load_tif("/content/gdrive/MyDrive/DatasetNCA/drivers/hn_ic_2000s.tif")))
river_drv = invert(normalize(load_tif("/content/gdrive/MyDrive/DatasetNCA/drivers/hn_water_acc.tif")))

# =====================================================================
# 3. BUILD THE 16-CHANNEL SEED TENSOR (X_seed) # NOTE: Renaming X_seed for prediction context
# =====================================================================
H, W = seed_2000.shape
hidden_channels = [np.zeros((H, W), dtype=np.float32) for _ in range(7)]

all_layers = [
    seed_2000.copy(),   # Channel 0: Dynamic Target (Starts as late 2000s layout)
    agri_drv,          # Channel 1: Dynamic
    forest_drv,        # Channel 2: Dynamic
    grass_drv,         # Channel 3: Dynamic
    barren_drv,        # Channel 4: Dynamic
    water_drv,         # Channel 5: Static Constraint (lc7)
    subway_drv,         # Channel 6: Static Driver (sws)
    highway_drv,        # Channel 7: Static Driver (ic)
    river_drv           # Channel 8: Static Constraint
] + hidden_channels     # Channels 9-15: Hidden Communication

# Stack and convert to PyTorch Tensor -> Shape: (1, 16, H, W)
X_seed_prediction = torch.tensor(np.stack(all_layers, axis=0), dtype=torch.float32).unsqueeze(0)


X_seed_prediction = X_seed_prediction.to(device)
# =====================================================================
# DIAGNOSTIC CHECK: PRINT LAYER STATISTICS
# =====================================================================
layer_names = [
    "Seed Map", "Agri DRV", "Forest DRV", "Grass DRV",
    "Barren DRV", "Water DRV", "Subway DRV", "Highway DRV", "River DRV"
]

print("\n--- 1990 INPUT TENSOR STATISTICS ---")
for i in range(9):  # Check the 10 physical layers
    layer_data = all_layers[i]
    print(f"{layer_names[i]:<12} | Min: {np.min(layer_data):.4f} | Max: {np.max(layer_data):.4f} | Mean: {np.mean(layer_data):.4f}")
print("------------------------------------\n")

# =====================================================================
# 3. RUN SIMULATION
# =====================================================================
eval_steps = 200
print(f"Running simulation for {eval_steps} steps...")

with torch.no_grad():
    # Pass the 2000 seed through the model rules
    final_state = model(X_seed_prediction, steps=eval_steps)

    # Extract Channel 0 (Predicted Urban Map) and push back to CPU numpy array
    raw_prediction = final_state[0, 0, :, :].cpu().numpy()

# Apply thresholding to create a clean binary map (1 for Urban, 0 for Non-Urban)
binary_prediction = (raw_prediction > 0.5).astype(np.float32)

# =====================================================================
# 4. ATTACH GEOGRAPHIC METADATA & EXPORT TIF
# =====================================================================
# Read your original 2000 map metadata to perfectly match GIS coordinates
reference_file = "/content/gdrive/MyDrive/DatasetNCA/validate/hn_ua_late1990s_mask.tif"

with rasterio.open(reference_file) as src:
    meta = src.meta.copy()

# Adjust profile configuration for the output file
meta.update({
    "driver": "GTiff",
    "height": binary_prediction.shape[0],
    "width": binary_prediction.shape[1],
    "count": 1,
    "dtype": "float32",
    "compress": "lzw"
})

# Save the predicted 2010 map
output_filepath = "outputs/predicted_urban.tif"
with rasterio.open(output_filepath, "w", **meta) as dest:
    dest.write(binary_prediction, 1)

print("-" * 50)
print(f"✅ Map generated successfully: {output_filepath}")

In [ ]:
import os
import numpy as np
import rasterio
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, cohen_kappa_score

# =====================================================================
# 1. LOAD THE THREE ESSENTIAL MAPS
# =====================================================================
print("Loading maps for validation...")

# Base  map (needed to isolate what actually changed)
with rasterio.open("/content/gdrive/MyDrive/DatasetNCA/inputs/hn_ua_late2000s_mask.tif") as src:
    base_map = src.read(1).astype(np.uint8)

# Actual  ground truth target
with rasterio.open("/content/gdrive/MyDrive/DatasetNCA/validate/hn_ua_late2010s_mask.tif") as src:
    actual_map = src.read(1).astype(np.uint8)

# Your model's prediction
with rasterio.open("outputs/predicted_urban.tif") as src:
    predicted_map = src.read(1).astype(np.uint8)

# Flatten arrays to 1D vectors for scikit-learn metrics
y_true = actual_map.flatten()
y_pred = predicted_map.flatten()

# =====================================================================
# 2. STANDARD CLASSIFICATION METRICS
# =====================================================================
accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, pos_label=1)
recall    = recall_score(y_true, y_pred, pos_label=1)
f1        = f1_score(y_true, y_pred, pos_label=1)
kappa     = cohen_kappa_score(y_true, y_pred)

# =====================================================================
# 3. URBAN GROWTH FIGURE OF MERIT (FoM)
# =====================================================================
# Isolate observed actual changes vs predicted changes from the 2000 baseline
observed_change  = (actual_map == 1) & (base_map == 0)
predicted_change = (predicted_map == 1) & (base_map == 0)

# Calculate Hits, Misses, and False Alarms
hits         = np.sum(observed_change & predicted_change)  # Predicted change correctly
misses       = np.sum(observed_change & ~predicted_change) # Failed to predict real growth
false_alarms = np.sum(~observed_change & predicted_change) # Predicted growth where none happened

# Figure of Merit Formula
fom = hits / (hits + misses + false_alarms) if (hits + misses + false_alarms) > 0 else 0.0

# =====================================================================
# 4. PRINT REPORT
# =====================================================================
print("\n" + "="*40)
print("       URBAN NCA VALIDATION REPORT       ")
print("="*40)
print(f"Overall Pixel Accuracy: {accuracy:.4f}")
print(f"Cohen's Kappa:          {kappa:.4f}")
print(f"F1-Score (Urban Class): {f1:.4f}")
print(f"Precision (Exactness):  {precision:.4f}")
print(f"Recall (Completeness):  {recall:.4f}")
print("-" * 40)
print("SPATIAL CHANGE ANALYSIS (Figure of Merit)")
print("-" * 40)
print(f"Correct Growth (Hits):        {hits} pixels")
print(f"Missed Growth (Misses):      {misses} pixels")
print(f"Wrong Growth (False Alarms):  {false_alarms} pixels")
print(f"👉 Figure of Merit (FoM):     {fom * 100:.2f}%")
print("="*40)

In [ ]:
import os
import torch
import rasterio
import numpy as np

# Leverage your excellent project structure!
# from src.datasets import X_seed # Removed as X_seed is already defined
# from src.models import UrbanNCA # Removed as UrbanNCA is already defined

# =====================================================================
# 1. SETUP DEVICE
# =====================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Loading model on: {device}")

# =====================================================================
# 2. INITIALIZE MODEL & LOAD TRAINED WEIGHTS
# =====================================================================
# Rebuild the model architecture from your src folder
model = UrbanNCA(channels=16).to(device)

# Load the saved .pth weights file from your outputs directory
model.load_state_dict(torch.load("outputs/trained_nca_model.pth", map_location=device))

# Set model to evaluation mode (turns off training mechanics)
model.eval()

# =====================================================================
# 1. HELPER FUNCTIONS
# =====================================================================
def load_tif(filepath):
    """Opens a single-band .tif file and reads it as a float32 numpy array."""
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Missing file: {filepath}")
    with rasterio.open(filepath) as src:
        return src.read(1).astype(np.float32)

def normalize(arr):
    min_val, max_val = np.min(arr), np.max(arr)
    if max_val == min_val: return np.zeros_like(arr)
    return (arr - min_val) / (max_val - min_val)

def invert(arr):
    return 1.0 - arr

def prep_distance_fixed(filepath_1990, filepath_2000):
    """Normalizes the 1990 map using the exact scale of the 2000 map to prevent distribution shift."""
    # 1. Get the exact scale from the 2000 training map
    arr_2000 = load_tif(filepath_2000)
    fixed_min = np.nanmin(arr_2000)
    fixed_max = np.nanmax(arr_2000)

    # 2. Load the 1990 map
    arr_1990 = load_tif(filepath_1990)

    # 3. Scale 1990 using the 2000 min/max
    if fixed_max == fixed_min:
        return np.zeros_like(arr_1990)

    normalized = (arr_1990 - fixed_min) / (fixed_max - fixed_min)

    # 4. Clip values just in case 1990 had longer distances (prevent negative or >1 values)
    normalized = np.clip(normalized, 0.0, 1.0)

    # 5. Invert it so roads are hotspots
    inverted = 1.0 - normalized
    inverted[np.isnan(inverted)] = 0.0
    return inverted

# =====================================================================
# 2. LOAD 2000 DATA (THE STARTING POINT & DRIVERS)
# =====================================================================
# The Seed: use the real 2000 urban map stored in the inputs folder.
seed_2000 = load_tif("/content/gdrive/MyDrive/DatasetNCA/inputs/hn_ua_late1990s_mask.tif")

# The Land Cover Drivers (lc1 through lc7)
agri_drv         = load_tif("/content/gdrive/MyDrive/DatasetNCA/drivers/hn_1990_lc2.tif")
forest_drv       = load_tif("/content/gdrive/MyDrive/DatasetNCA/drivers/hn_1990_lc3.tif")
grass_drv        = load_tif("/content/gdrive/MyDrive/DatasetNCA/drivers/hn_1990_lc4.tif")
barren_drv       = load_tif("/content/gdrive/MyDrive/DatasetNCA/drivers/hn_1990_lc6.tif")
water_drv        = load_tif("/content/gdrive/MyDrive/DatasetNCA/drivers/hn_1990_lc7.tif")

# Infrastructure Drivers
subway_drv  = invert(normalize(load_tif("/content/gdrive/MyDrive/DatasetNCA/drivers/hn_sws_1990s.tif")))
highway_drv = invert(normalize(load_tif("/content/gdrive/MyDrive/DatasetNCA/drivers/hn_ic_1990s.tif")))
river_drv = invert(normalize(load_tif("/content/gdrive/MyDrive/DatasetNCA/drivers/hn_water_acc.tif")))

# =====================================================================
# 3. BUILD THE 16-CHANNEL SEED TENSOR (X_seed) # NOTE: Renaming X_seed for prediction context
# =====================================================================
H, W = seed_2000.shape
hidden_channels = [np.zeros((H, W), dtype=np.float32) for _ in range(7)]

all_layers = [
    seed_2000.copy(),   # Channel 0: Dynamic Target (Starts as late 2000s layout)
    agri_drv,          # Channel 1: Dynamic
    forest_drv,        # Channel 2: Dynamic
    grass_drv,         # Channel 3: Dynamic
    barren_drv,        # Channel 4: Dynamic
    water_drv,         # Channel 5: Static Constraint (lc7)
    subway_drv,         # Channel 6: Static Driver (sws)
    highway_drv,        # Channel 7: Static Driver (ic)
    river_drv           # Channel 8: Static Constraint
] + hidden_channels     # Channels 9-15: Hidden Communication

# Stack and convert to PyTorch Tensor -> Shape: (1, 16, H, W)
X_seed_prediction = torch.tensor(np.stack(all_layers, axis=0), dtype=torch.float32).unsqueeze(0)


X_seed_prediction = X_seed_prediction.to(device)
# =====================================================================
# DIAGNOSTIC CHECK: PRINT LAYER STATISTICS
# =====================================================================
layer_names = [
    "Seed Map", "Agri DRV", "Forest DRV", "Grass DRV",
    "Barren DRV", "Water DRV", "Subway DRV", "Highway DRV", "River DRV"
]

print("\n--- 1990 INPUT TENSOR STATISTICS ---")
for i in range(9):  # Check the 10 physical layers
    layer_data = all_layers[i]
    print(f"{layer_names[i]:<12} | Min: {np.min(layer_data):.4f} | Max: {np.max(layer_data):.4f} | Mean: {np.mean(layer_data):.4f}")
print("------------------------------------\n")

# =====================================================================
# 3. RUN SIMULATION
# =====================================================================
eval_steps = 200
print(f"Running simulation for {eval_steps} steps...")

with torch.no_grad():
    # Pass the 2000 seed through the model rules
    final_state = model(X_seed_prediction, steps=eval_steps)

    # Extract Channel 0 (Predicted Urban Map) and push back to CPU numpy array
    raw_prediction = final_state[0, 0, :, :].cpu().numpy()

# Apply thresholding to create a clean binary map (1 for Urban, 0 for Non-Urban)
binary_prediction = (raw_prediction > 0.5).astype(np.float32)

# =====================================================================
# 4. ATTACH GEOGRAPHIC METADATA & EXPORT TIF
# =====================================================================
# Read your original 2000 map metadata to perfectly match GIS coordinates
reference_file = "/content/gdrive/MyDrive/DatasetNCA/validate/hn_ua_late1990s_mask.tif"

with rasterio.open(reference_file) as src:
    meta = src.meta.copy()

# Adjust profile configuration for the output file
meta.update({
    "driver": "GTiff",
    "height": binary_prediction.shape[0],
    "width": binary_prediction.shape[1],
    "count": 1,
    "dtype": "float32",
    "compress": "lzw"
})

# Save the predicted 2010 map
output_filepath = "outputs/predicted_urban.tif"
with rasterio.open(output_filepath, "w", **meta) as dest:
    dest.write(binary_prediction, 1)

print("-" * 50)
print(f"✅ Map generated successfully: {output_filepath}")


import os
import numpy as np
import rasterio
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, cohen_kappa_score

# =====================================================================
# 1. LOAD THE THREE ESSENTIAL MAPS
# =====================================================================
print("Loading maps for validation...")

# Base  map (needed to isolate what actually changed)
with rasterio.open("/content/gdrive/MyDrive/DatasetNCA/inputs/hn_ua_late1990s_mask.tif") as src:
    base_map = src.read(1).astype(np.uint8)

# Actual  ground truth target
with rasterio.open("/content/gdrive/MyDrive/DatasetNCA/validate/hn_ua_late2000s_mask.tif") as src:
    actual_map = src.read(1).astype(np.uint8)

# Your model's prediction
with rasterio.open("outputs/predicted_urban.tif") as src:
    predicted_map = src.read(1).astype(np.uint8)

# Flatten arrays to 1D vectors for scikit-learn metrics
y_true = actual_map.flatten()
y_pred = predicted_map.flatten()

# =====================================================================
# 2. STANDARD CLASSIFICATION METRICS
# =====================================================================
accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, pos_label=1)
recall    = recall_score(y_true, y_pred, pos_label=1)
f1        = f1_score(y_true, y_pred, pos_label=1)
kappa     = cohen_kappa_score(y_true, y_pred)

# =====================================================================
# 3. URBAN GROWTH FIGURE OF MERIT (FoM)
# =====================================================================
# Isolate observed actual changes vs predicted changes from the 2000 baseline
observed_change  = (actual_map == 1) & (base_map == 0)
predicted_change = (predicted_map == 1) & (base_map == 0)

# Calculate Hits, Misses, and False Alarms
hits         = np.sum(observed_change & predicted_change)  # Predicted change correctly
misses       = np.sum(observed_change & ~predicted_change) # Failed to predict real growth
false_alarms = np.sum(~observed_change & predicted_change) # Predicted growth where none happened

# Figure of Merit Formula
fom = hits / (hits + misses + false_alarms) if (hits + misses + false_alarms) > 0 else 0.0

# =====================================================================
# 4. PRINT REPORT
# =====================================================================
print("\n" + "="*40)
print("       URBAN NCA VALIDATION REPORT       ")
print("="*40)
print(f"Overall Pixel Accuracy: {accuracy:.4f}")
print(f"Cohen's Kappa:          {kappa:.4f}")
print(f"F1-Score (Urban Class): {f1:.4f}")
print(f"Precision (Exactness):  {precision:.4f}")
print(f"Recall (Completeness):  {recall:.4f}")
print("-" * 40)
print("SPATIAL CHANGE ANALYSIS (Figure of Merit)")
print("-" * 40)
print(f"Correct Growth (Hits):        {hits} pixels")
print(f"Missed Growth (Misses):      {misses} pixels")
print(f"Wrong Growth (False Alarms):  {false_alarms} pixels")
print(f"👉 Figure of Merit (FoM):     {fom * 100:.2f}%")
print("="*40)


